In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [7]:
# =========================
# CONFIG
# =========================

DATA_PATH = "../../data/features/m5_features"
STORE = "CA_1"
# N_ITEMS = 100        # subset pequeño
# LOOKBACK = 28

In [8]:
# =========================
# LOAD DATA
# =========================

df = pd.read_parquet(
    DATA_PATH,
    filters=[("store_id", "==", STORE)]
)

df["date"] = pd.to_datetime(df["date"])

# seleccionar subset de items
top_items = df["item_id"].value_counts().head(N_ITEMS).index
df = df[df["item_id"].isin(top_items)]

df = df.sort_values(["item_id", "date"])

In [9]:
# =========================
# PREP SERIES
# =========================

def create_sequences(series, lookback):
    X, y = [], []
    for i in range(len(series) - lookback):
        X.append(series[i:i+lookback])
        y.append(series[i+lookback])
    return np.array(X), np.array(y)

# construir dataset LSTM
X_list, y_list = [], []

for item in df["item_id"].unique():
    s = df[df["item_id"] == item]["sales"].values
    
    if len(s) < LOOKBACK + 1:
        continue
    
    X_seq, y_seq = create_sequences(s, LOOKBACK)
    
    X_list.append(X_seq)
    y_list.append(y_seq)

X_lstm = np.concatenate(X_list)
y_lstm = np.concatenate(y_list)

# reshape para LSTM
X_lstm = X_lstm.reshape((X_lstm.shape[0], X_lstm.shape[1], 1))

# split simple
split = int(0.8 * len(X_lstm))

X_train_lstm, X_test_lstm = X_lstm[:split], X_lstm[split:]
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]


In [10]:
# =========================
# LSTM MODEL
# =========================

model_lstm = Sequential([
    LSTM(32, input_shape=(LOOKBACK, 1)),
    Dense(1)
])

model_lstm.compile(
    optimizer="adam",
    loss="mse"
)

model_lstm.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=5,
    batch_size=256,
    verbose=1
)

y_pred_lstm = model_lstm.predict(X_test_lstm).flatten()

rmse_lstm = np.sqrt(mean_squared_error(y_test_lstm, y_pred_lstm))
mae_lstm = mean_absolute_error(y_test_lstm, y_pred_lstm)

print("\nLSTM Results:")
print(f"RMSE: {rmse_lstm:.4f}")
print(f"MAE: {mae_lstm:.4f}")

Epoch 1/5


C:\Users\Titanio\anaconda3\envs\tfm\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


590/590 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - loss: 6.5969
Epoch 2/5
590/590 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - loss: 5.6497
Epoch 3/5
590/590 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - loss: 5.4441
Epoch 4/5
590/590 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 5.3464
Epoch 5/5
590/590 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - loss: 5.2823
1179/1179 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step

LSTM Results:
RMSE: 0.7822
MAE: 0.4532


In [11]:
# =========================
# LIGHTGBM BASELINE
# =========================

# features simples (lags)
df["lag_1"] = df.groupby("item_id")["sales"].shift(1)
df["lag_7"] = df.groupby("item_id")["sales"].shift(7)
df["lag_28"] = df.groupby("item_id")["sales"].shift(28)

df = df.dropna()

features = ["lag_1", "lag_7", "lag_28"]

X = df[features]
y = df["sales"]

split = int(0.8 * len(X))

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model_lgb = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)

model_lgb.fit(X_train, y_train)

y_pred_lgb = model_lgb.predict(X_test)

rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
mae_lgb = mean_absolute_error(y_test, y_pred_lgb)

print("\nLightGBM Results:")
print(f"RMSE: {rmse_lgb:.4f}")
print(f"MAE: {mae_lgb:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41
[LightGBM] [Info] Number of data points in the train set: 253, number of used features: 3
[LightGBM] [Info] Start training from score 1.837945
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best